# README
## Purpose
Clean and normalize GSIB headlines for modeling and evaluation.
## Inputs
- `Data/eodhd_sample_clean.csv`
## Outputs
- Preprocessed GSIB model-ready table (saved in notebook workflow as `Data/eodhd_sample_model_input.csv`).
## Notes
Includes language filtering and entity masking to reduce ticker-dominated topics.

Imports

In [1]:
import re
import unicodedata
import numpy as np
import pandas as pd
from langdetect import detect_langs, LangDetectException

## Loading Data

In [2]:
data = pd.read_csv('Data/eodhd_sample_clean.csv')
print('Raw shape:', data.shape)
display(data.head(5))

Raw shape: (15303, 4)


,stock,headline,date,date_only
0,JPM,Berkshire shares struggle into annual meeting....,2026-05-01 11:12:47+00:00,2026-05-01
1,JPM,Social Buzz: Wallstreetbets Stocks Mostly Adva...,2026-05-01 10:36:54+00:00,2026-05-01
2,JPM,Gold Declines as Trump Vows to Maintain Pressu...,2026-05-01 09:47:00+00:00,2026-05-01
3,JPM,Aluminum Gains as Trump Vows to Maintain Naval...,2026-05-01 08:58:25+00:00,2026-05-01
4,JPM,Jane Fraser Turned Citigroup Inside Out. Now C...,2026-05-01 05:00:00+00:00,2026-05-01


## Preprocessing configuration and cleaning

In [3]:
# Choose modeling text strategy: 'keep', 'mask', or 'remove'
ENTITY_MODE = 'mask'

# Language detection: keep only English headlines
LANGUAGE_FILTER = 'en'  # ISO 639-1 language code for English
MIN_CONFIDENCE_LANG = 0.5  # Minimum confidence threshold for English detection

# Full ticker universe for masking
bank_tickers = [
    'JPM', 'BAC', 'C', 'HSBC', 'IDCBY', 'GS', 'BNPQY', 'UBS', 'ACGBY', 'BACHY',
    'CICHY', 'BCS', 'MUFG', 'MS', 'WFC', 'BK', 'STT', 'DB', 'ING', 'SAN', 'RY',
    'TD', 'SCBFF', 'SCGLY', 'MFG', 'SMFG', 'BCMXY', 'GCRLY', 'BPCE'
 ]
BANK_TICKERS = sorted({ticker.upper().strip() for ticker in bank_tickers})

# Optional: map ticker symbols to common aliases to reduce entity-dominated topics
STOCK_ALIAS_MAP = {
    'JPM': [r'jpmorgan', r'j\.p\.\s*morgan'],
    'BAC': [r'bank\s+of\s+america', r'bofa'],
    'WFC': [r'wells\s+fargo'],
    'C': [r'citigroup', r'\bciti\b'],
    'GS': [r'goldman\s+sachs'],
    'MS': [r'morgan\s+stanley'],
    'HSBC': [r'\bhsbc\b'],
    'SAN': [r'santander', r'banco\s+santander'],
    'BCS': [r'barclays'],
    'DB': [r'deutsche\s+bank'],
    'UBS': [r'\bubs\b'],
    'RY': [r'royal\s+bank\s+of\s+canada', r'\brbc\b'],
    'TD': [r'toronto\-dominion', r'toronto\s+dominion'],
    'SCBFF': [r'standard\s+chartered', r'\bstanchart\b'],
    'SCGLY': [r'societe\s+generale', r'société\s+générale', r'\bsocgen\b'],
    'MFG': [r'mizuho\s+financial\s+group', r'\bmizuho\b'],
    'SMFG': [r'sumitomo\s+mitsui', r'\bsmbc\b'],
    'MUFG': [r'mitsubishi\s+ufj', r'\bufj\b'],
    'BK': [r'bank\s+of\s+new\s+york\s+mellon', r'\bbny\b'],
    'STT': [r'state\s+street'],
    'IDCBY': [r'industrial\s+and\s+commercial\s+bank\s+of\s+china', r'\bicbc\b'],
    'ACGBY': [r'agricultural\s+bank\s+of\s+china'],
    'CICHY': [r'china\s+construction\s+bank'],
    'BACHY': [r'bank\s+of\s+china'],
    'BCMXY': [r'bank\s+of\s+communications'],
    'ING': [r'\bing\b\s+groep'],
    'BNPQY': [r'bnp\s+paribas'],
    'GCRLY': [r'credit\s+agricole', r'crédit\s+agricole'],
    'BPCE': [r'\bbpce\b']
}

print('ENTITY_MODE:', ENTITY_MODE)
print('LANGUAGE_FILTER:', LANGUAGE_FILTER)
print('MIN_CONFIDENCE_LANG:', MIN_CONFIDENCE_LANG)
print('Ticker count configured:', len(BANK_TICKERS))

ENTITY_MODE: mask
LANGUAGE_FILTER: en
MIN_CONFIDENCE_LANG: 0.5
Ticker count configured: 29


In [4]:
def is_english(text: str, lang_code: str = 'en', min_confidence: float = 0.5) -> bool:
    """
    Detect if text is in the specified language using langdetect.
    Returns True if the language matches with sufficient confidence.
    """
    if not text or len(text.strip()) < 3:
        return False
    
    try:
        detected_langs = detect_langs(text)
        for lang_prob in detected_langs:
            if lang_prob.lang == lang_code and lang_prob.prob >= min_confidence:
                return True
        return False
    except LangDetectException:
        # If detection fails, assume non-English
        return False

def normalize_text(text: str) -> str:
    text = '' if pd.isna(text) else str(text)
    text = unicodedata.normalize('NFKC', text)
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'\s+\|\s*(Reuters|Bloomberg|CNBC|Yahoo|MarketWatch|Barrons?)\b.*$', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'\s+-\s*(Reuters|Bloomberg|CNBC|Yahoo|MarketWatch|Barrons?)\b.*$', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def _mask_ticker_forms(text: str, ticker: str, replacement: str) -> str:
    out = text
    t = re.escape(ticker)
    out = re.sub(rf'\${t}\b', replacement, out)
    out = re.sub(rf'\b(?:NYSE|NASDAQ|AMEX|TSX|LSE|HKEX|TSE)\s*:\s*{t}\b', replacement, out, flags=re.IGNORECASE)
    out = re.sub(rf'\({t}\)', replacement, out)
    out = re.sub(rf'(?<!_)\b{t}\b(?!_)', replacement, out)
    return out

def handle_stock_entities(text: str, stock: str, mode: str = 'mask') -> str:
    cleaned = text
    replacement_ticker = ' __TICKER__ ' if mode == 'mask' else ' '
    replacement_company = ' __COMPANY__ ' if mode == 'mask' else ' '

    if mode in ('mask', 'remove'):
        # Generic cashtags and exchange-prefixed tickers
        cleaned = re.sub(r'\$[A-Z]{1,7}\b', replacement_ticker, cleaned)
        cleaned = re.sub(r'\b(?:NYSE|NASDAQ|AMEX|TSX|LSE|HKEX|TSE)\s*:\s*[A-Z0-9.]{1,10}\b', replacement_ticker, cleaned, flags=re.IGNORECASE)

        # Mask all configured bank tickers globally
        for ticker in BANK_TICKERS:
            cleaned = _mask_ticker_forms(cleaned, ticker, replacement_ticker)

        # Row stock ticker (extra safety)
        row_stock = '' if pd.isna(stock) else str(stock).upper().strip()
        if row_stock:
            cleaned = _mask_ticker_forms(cleaned, row_stock, replacement_ticker)

        # Mask aliases/names
        for ticker, alias_patterns in STOCK_ALIAS_MAP.items():
            for pattern in alias_patterns:
                cleaned = re.sub(pattern, replacement_company, cleaned, flags=re.IGNORECASE)

    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    return cleaned

def preprocess_headline(text: str, stock: str, mode: str = 'mask') -> str:
    cleaned = normalize_text(text)
    if mode in ('mask', 'remove', 'keep') and mode != 'keep':
        cleaned = handle_stock_entities(cleaned, stock, mode=mode)
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    return cleaned

## Apply preprocessing

In [5]:
required_cols = ['stock', 'headline', 'date']
missing = [c for c in required_cols if c not in data.columns]
if missing:
    raise ValueError(f'Missing required columns: {missing}')

# Short-text filtering thresholds
MIN_HEADLINE_CHARS = 20
MIN_HEADLINE_TOKENS = 4

df = data.copy()
df = df.dropna(subset=['stock', 'headline', 'date']).copy()
df['stock'] = df['stock'].astype(str).str.upper().str.strip()
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df = df.dropna(subset=['date'])
df['date_only'] = df['date'].dt.date

# Preserve raw headline and create variants
df['headline_raw'] = df['headline'].astype(str).str.strip()

# Filter for English headlines BEFORE preprocessing
print('Detecting language of headlines...')
df['is_english'] = df['headline_raw'].apply(
    lambda x: is_english(x, lang_code=LANGUAGE_FILTER, min_confidence=MIN_CONFIDENCE_LANG)
)
rows_before_lang = len(df)
df = df[df['is_english']].copy()
rows_after_lang = len(df)
print(f'Language filter: removed {rows_before_lang - rows_after_lang} non-English headlines')
print(f'Remaining rows: {rows_after_lang}')

df['headline_clean_keep'] = df.apply(lambda r: preprocess_headline(r['headline_raw'], r['stock'], mode='keep'), axis=1)
df['headline_clean_mask'] = df.apply(lambda r: preprocess_headline(r['headline_raw'], r['stock'], mode='mask'), axis=1)
df['headline_clean_remove'] = df.apply(lambda r: preprocess_headline(r['headline_raw'], r['stock'], mode='remove'), axis=1)

if ENTITY_MODE == 'keep':
    df['headline_model'] = df['headline_clean_keep']
elif ENTITY_MODE == 'remove':
    df['headline_model'] = df['headline_clean_remove']
else:
    df['headline_model'] = df['headline_clean_mask']

# Remove empty model headlines
df['headline_model'] = df['headline_model'].fillna('').str.strip()
df = df[df['headline_model'].str.len() > 0].copy()

# Remove very short headlines (chars + token count)
rows_before_short_filter = len(df)
headline_token_count = df['headline_model'].str.split().str.len()
short_mask = (df['headline_model'].str.len() < MIN_HEADLINE_CHARS) | (headline_token_count < MIN_HEADLINE_TOKENS)
removed_short = int(short_mask.sum())
df = df[~short_mask].copy()

# Remove duplicates after filtering
df = df.drop_duplicates(subset=['stock', 'date', 'headline_model']).sort_values('date').reset_index(drop=True)

print(f'Removed short headlines: {removed_short} (min chars={MIN_HEADLINE_CHARS}, min tokens={MIN_HEADLINE_TOKENS})')
print('Processed shape:', df.shape)
display(df[['stock', 'date', 'date_only', 'headline_raw', 'headline_model']].head(10))

Detecting language of headlines...
Language filter: removed 666 non-English headlines
Remaining rows: 14637
Removed short headlines: 33 (min chars=20, min tokens=4)
Processed shape: (14604, 10)


,stock,date,date_only,headline_raw,headline_model
0,SAN,2025-05-01 07:02:46+00:00,2025-05-01,Banco Santander SA (SAN) Q1 2025 Earnings Call...,Banco __COMPANY__ SA __TICKER__ Q1 2025 Earnin...
1,MUFG,2025-05-01 07:48:39+00:00,2025-05-01,"Yen extends drop, BOJ hike bets fall as Ueda v...","Yen extends drop, BOJ hike bets fall as Ueda v..."
2,BK,2025-05-01 08:05:14+00:00,2025-05-01,BNY gets licence for Saudi regional HQ as glob...,__COMPANY__ gets licence for Saudi regional HQ...
3,SAN,2025-05-01 08:50:00+00:00,2025-05-01,"Zacks.com featured highlights Centene, Pediatr...","Zacks.com featured highlights Centene, Pediatr..."
4,BK,2025-05-01 10:15:33+00:00,2025-05-01,Morgan Stanley Plans to Offer Crypto Trading t...,__COMPANY__ Plans to Offer Crypto Trading to E...
5,TD,2025-05-01 10:30:00+00:00,2025-05-01,3 Brilliant High-Yield Stocks to Buy Now and H...,3 Brilliant High-Yield Stocks to Buy Now and H...
6,BK,2025-05-01 13:00:19+00:00,2025-05-01,Possible Bearish Signals With Bank of New York...,Possible Bearish Signals With __COMPANY__ Insi...
7,MUFG,2025-05-01 14:45:27+00:00,2025-05-01,Mitsubishi UFJ downgraded to Neutral from Buy ...,__COMPANY__ downgraded to Neutral from Buy at ...
8,MFG,2025-05-01 14:45:27+00:00,2025-05-01,Mitsubishi UFJ downgraded to Neutral from Buy ...,__COMPANY__ downgraded to Neutral from Buy at ...
9,TD,2025-05-01 15:00:00+00:00,2025-05-01,Media Advisory - TD Bank Group to release seco...,Media Advisory - __TICKER__ Bank Group to rele...


## Quality checks

## Language Detection Results

In [6]:
print(f'\n✅ Language Filter Applied: {LANGUAGE_FILTER} only')
print(f'   Minimum confidence threshold: {MIN_CONFIDENCE_LANG}')
print(f'   Non-English rows removed: {rows_before_lang - rows_after_lang}')
print(f'   Final dataset (English only): {len(df)} rows')
print(f'\nLanguage detection successfully filtered dataset.')
if rows_before_lang > rows_after_lang:
    pct_removed = 100 * (rows_before_lang - rows_after_lang) / rows_before_lang
    print(f'   Percentage removed: {pct_removed:.1f}%')


✅ Language Filter Applied: en only
   Minimum confidence threshold: 0.5
   Non-English rows removed: 666
   Final dataset (English only): 14604 rows

Language detection successfully filtered dataset.
   Percentage removed: 4.4%


In [7]:
summary = {
    'rows': len(df),
    'unique_stocks': int(df['stock'].nunique()),
    'date_min': df['date'].min(),
    'date_max': df['date'].max(),
    'empty_model_headlines': int((df['headline_model'].str.len() == 0).sum())
}
print(summary)

print('\nTop stocks by article count:')
display(df['stock'].value_counts().head(15).to_frame('n_articles'))

df['raw_len'] = df['headline_raw'].str.len()
df['model_len'] = df['headline_model'].str.len()
print('\nHeadline length stats (raw vs model):')
display(df[['raw_len', 'model_len']].describe().T)

print('\nExamples of transformations:')
display(df[['stock', 'headline_raw', 'headline_clean_mask', 'headline_clean_remove', 'headline_model']].head(10))

{'rows': 14604, 'unique_stocks': 27, 'date_min': Timestamp('2025-05-01 07:02:46+0000', tz='UTC'), 'date_max': Timestamp('2026-05-01 12:55:00+0000', tz='UTC'), 'empty_model_headlines': 0}

Top stocks by article count:


,n_articles
stock,
C,981
MS,981
BCS,980
UBS,977
WFC,968
GS,959
BAC,953
JPM,952
HSBC,922



Headline length stats (raw vs model):


,count,mean,std,min,25%,50%,75%,max
raw_len,14604.0,73.145508,22.286666,19.0,58.0,70.0,86.0,296.0
model_len,14604.0,74.597371,22.580403,22.0,60.0,71.0,88.0,296.0



Examples of transformations:


,stock,headline_raw,headline_clean_mask,headline_clean_remove,headline_model
0,SAN,Banco Santander SA (SAN) Q1 2025 Earnings Call...,Banco __COMPANY__ SA __TICKER__ Q1 2025 Earnin...,Banco SA Q1 2025 Earnings Call Highlights: Rec...,Banco __COMPANY__ SA __TICKER__ Q1 2025 Earnin...
1,MUFG,"Yen extends drop, BOJ hike bets fall as Ueda v...","Yen extends drop, BOJ hike bets fall as Ueda v...","Yen extends drop, BOJ hike bets fall as Ueda v...","Yen extends drop, BOJ hike bets fall as Ueda v..."
2,BK,BNY gets licence for Saudi regional HQ as glob...,__COMPANY__ gets licence for Saudi regional HQ...,gets licence for Saudi regional HQ as global b...,__COMPANY__ gets licence for Saudi regional HQ...
3,SAN,"Zacks.com featured highlights Centene, Pediatr...","Zacks.com featured highlights Centene, Pediatr...","Zacks.com featured highlights Centene, Pediatr...","Zacks.com featured highlights Centene, Pediatr..."
4,BK,Morgan Stanley Plans to Offer Crypto Trading t...,__COMPANY__ Plans to Offer Crypto Trading to E...,Plans to Offer Crypto Trading to E*Trade Clients,__COMPANY__ Plans to Offer Crypto Trading to E...
5,TD,3 Brilliant High-Yield Stocks to Buy Now and H...,3 Brilliant High-Yield Stocks to Buy Now and H...,3 Brilliant High-Yield Stocks to Buy Now and H...,3 Brilliant High-Yield Stocks to Buy Now and H...
6,BK,Possible Bearish Signals With Bank of New York...,Possible Bearish Signals With __COMPANY__ Insi...,Possible Bearish Signals With Insiders Disposi...,Possible Bearish Signals With __COMPANY__ Insi...
7,MUFG,Mitsubishi UFJ downgraded to Neutral from Buy ...,__COMPANY__ downgraded to Neutral from Buy at ...,downgraded to Neutral from Buy at,__COMPANY__ downgraded to Neutral from Buy at ...
8,MFG,Mitsubishi UFJ downgraded to Neutral from Buy ...,__COMPANY__ downgraded to Neutral from Buy at ...,downgraded to Neutral from Buy at,__COMPANY__ downgraded to Neutral from Buy at ...
9,TD,Media Advisory - TD Bank Group to release seco...,Media Advisory - __TICKER__ Bank Group to rele...,Media Advisory - Bank Group to release second-...,Media Advisory - __TICKER__ Bank Group to rele...


In [8]:
# Coverage check: any configured bank tickers still visible in headline_model?
remaining_counts = {}
for ticker in BANK_TICKERS:
    pattern = rf'(?<!_)\b{re.escape(ticker)}\b(?!_)'
    count = int(df['headline_model'].astype(str).str.contains(pattern, regex=True).sum())
    if count > 0:
        remaining_counts[ticker] = count

print('\nRemaining visible configured tickers in headline_model:', len(remaining_counts))
if remaining_counts:
    display(pd.DataFrame(sorted(remaining_counts.items(), key=lambda x: x[1], reverse=True), columns=['ticker', 'count']).head(20))
else:
    print('✅ All configured bank tickers are masked/removed from headline_model.')


Remaining visible configured tickers in headline_model: 0
✅ All configured bank tickers are masked/removed from headline_model.


## Save outputs for downstream pipelines

In [9]:
output_main = 'Data/eodhd_sample_preprocessed.csv'
output_model = 'Data/eodhd_sample_model_input.csv'

df.to_csv(output_main, index=False)

model_df = df[['stock', 'date', 'date_only', 'headline_model', 'headline_raw']].copy()
model_df = model_df.rename(columns={'headline_model': 'headline'})
model_df.to_csv(output_model, index=False)

print('Saved:')
print(f'  {output_main} -> {df.shape}')
print(f'  {output_model} -> {model_df.shape}')

Saved:
  Data/eodhd_sample_preprocessed.csv -> (14604, 12)
  Data/eodhd_sample_model_input.csv -> (14604, 5)
